In [1]:
import torch, torch.nn as nn
import torchvision, torchvision.transforms as transforms

# I. Preparing dataset

In [2]:
transform = transforms.Compose([transforms.Pad(4),
                                transforms.RandomHorizontalFlip(),
                                transforms.RandomCrop(32),
                                transforms.ToTensor()])
batch_size = 32

# CIFAR-10 dataset
train_dataset = torchvision.datasets.CIFAR10(root='../../data/', train=True, transform=transform, download=True)

test_dataset = torchvision.datasets.CIFAR10(root='../../data/', train=False, transform=transforms.ToTensor())

# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

100%|██████████| 170M/170M [00:04<00:00, 36.0MB/s]


In [3]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# II. Model

## 1. Model definition

In [4]:
def conv3x3(in_channels, out_channels, stride=1):
    return nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)

In [5]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet, self).__init__()

        self.conv = conv3x3(3, 16)
        self.bn = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        self.in_channels = 16

        # 3 continuos Resnet blocks
        self.layer1 = self.make_layer(block, 16, layers[0])
        self.layer2 = self.make_layer(block, 32, layers[1], 2)
        self.layer3 = self.make_layer(block, 64, layers[2], 2)
        self.avg_pool = nn.AvgPool2d(8)

        # Fully connected
        self.fc = nn.Linear(64, num_classes)

    def make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if (stride != 1) or (self.in_channels != out_channels):
            downsample = nn.Sequential(conv3x3(self.in_channels, out_channels, stride=stride),
                                       nn.BatchNorm2d(out_channels))

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels

        for i in range(1, blocks): layers.append(block(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        # x: [batch_size, 3, 32, 32]
        out = self.conv(x)
        out = self.bn(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)

        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [6]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = conv3x3(in_channels, out_channels, stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(out_channels, out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample


    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

## 2. Training model

In [7]:
# Hyper-parameters
num_epochs = 25
learning_rate = 0.001
model = ResNet(ResidualBlock, [2, 2, 2]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [8]:
decay = 0
model.train()
for epoch in range(num_epochs):

    # Decay the learning rate every 20 epochs
    if (epoch+1) % 20 == 0:
        decay+=1
        optimizer.param_groups[0]['lr'] = learning_rate * (0.5**decay)
        print("The new learning rate is {}".format(optimizer.param_groups[0]['lr']))

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 100 == 0:
            print ("Epoch [{}/{}], Step [{}/{}] Loss: {:.4f}"
                   .format(epoch+1, num_epochs, i+1, len(train_loader), loss.item()))

Epoch [1/25], Step [100/1563] Loss: 1.6520
Epoch [1/25], Step [200/1563] Loss: 1.7212
Epoch [1/25], Step [300/1563] Loss: 1.4636
Epoch [1/25], Step [400/1563] Loss: 1.3040
Epoch [1/25], Step [500/1563] Loss: 1.3597
Epoch [1/25], Step [600/1563] Loss: 1.4014
Epoch [1/25], Step [700/1563] Loss: 1.4243
Epoch [1/25], Step [800/1563] Loss: 1.1861
Epoch [1/25], Step [900/1563] Loss: 1.1489
Epoch [1/25], Step [1000/1563] Loss: 1.4491
Epoch [1/25], Step [1100/1563] Loss: 1.1661
Epoch [1/25], Step [1200/1563] Loss: 1.1643
Epoch [1/25], Step [1300/1563] Loss: 1.1508
Epoch [1/25], Step [1400/1563] Loss: 1.0302
Epoch [1/25], Step [1500/1563] Loss: 1.0379
Epoch [2/25], Step [100/1563] Loss: 1.2454
Epoch [2/25], Step [200/1563] Loss: 0.8327
Epoch [2/25], Step [300/1563] Loss: 1.0799
Epoch [2/25], Step [400/1563] Loss: 0.7916
Epoch [2/25], Step [500/1563] Loss: 0.9241
Epoch [2/25], Step [600/1563] Loss: 1.0431
Epoch [2/25], Step [700/1563] Loss: 0.8607
Epoch [2/25], Step [800/1563] Loss: 1.1730
Epoch

# III. Testing

In [9]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the model on the test images: {} %'.format(100 * correct / total))

Accuracy of the model on the test images: 87.19 %
